# Phase 40 — LIME Explainability

**Phases covered:** 40.1 LIME Explainer · 40.2 SHAP vs LIME Correlation · 40.3 LIME Stability

**Research Paper Connection:**
- §4.4 Explainability: LIME implementation details
- Table 8: LIME-SHAP Spearman ρ per model
- Answers **RQ4**: Which XAI method is most analyst-actionable?

**Key Concept — LIME vs SHAP:**
| Property | SHAP | LIME |
|---|---|---|
| Approach | Exact Shapley values | Local linear approximation |
| Speed | Slow for large feature sets | Fast (5000 perturbations ≈ 2–5s) |
| Consistency | Deterministic | Stochastic (fix seed=42 always!) |
| Faithfulness | Mathematically exact | Approximation — check R² |


In [ ]:
import sys
!{sys.executable} -m pip install lime shap scipy scikit-learn xgboost -q  # noqa
import warnings; warnings.filterwarnings("ignore")


In [ ]:
import os, mlflow
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5500")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow: {MLFLOW_TRACKING_URI}")


---
## Subphase 40.1 — LIME Explainer Implementation

We load the XGBoost Champion and create a `XAIGuardLIMEExplainer` instance.
The explainer is initialised **once** on the training distribution — reused for all predictions.


In [ ]:
import sys, os
ML_SRC = os.path.abspath("../../src")
if ML_SRC not in sys.path: sys.path.insert(0, ML_SRC)

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from xai.lime_explainer import XAIGuardLIMEExplainer, LIMEExplanation
from xai.stability import LIMEStabilityTester, SHAPStabilityTester, StabilityReport
print("✅ XAI modules imported from ml/src/")


In [ ]:
DATA_DIR   = os.path.abspath("../../data/processed")
MODEL_PATH = os.path.abspath("../../artifacts/models/xgboost_champion.json")

USE_REAL = os.path.exists(MODEL_PATH) and os.path.exists(os.path.join(DATA_DIR, "X_train.npy"))

if USE_REAL:
    import xgboost as xgb
    model = xgb.XGBClassifier()
    model.load_model(MODEL_PATH)
    X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
    X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
    y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))
    print(f"✅ Real model loaded | X_train: {X_train.shape}")
else:
    print("⚠️  No model found → synthetic demo")
    from sklearn.datasets import make_classification
    from sklearn.ensemble import GradientBoostingClassifier
    X_train, y_train = make_classification(n_samples=2000, n_features=20, n_informative=10, n_classes=2, random_state=42)
    X_test,  y_test  = make_classification(n_samples=500,  n_features=20, n_informative=10, n_classes=2, random_state=99)
    model = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)
    print("   Using synthetic data — run notebook 21 first for real results.")

N_FEATURES = X_train.shape[1]
feature_names = [f"feature_{i}" for i in range(N_FEATURES)]


In [ ]:
# Build the LIME explainer — expensive once, cheap per prediction after
lime_explainer = XAIGuardLIMEExplainer(
    X_train=X_train,
    feature_names=feature_names,
    n_perturbations=5000,   # ← minimum for 50+ feature spaces
    random_state=42,        # ← ALWAYS fixed in production
    class_names=["BENIGN", "ATTACK"],
)
print("✅ XAIGuardLIMEExplainer initialised")
print(f"   Background samples: {X_train.shape[0]}")
print(f"   Features: {N_FEATURES}")
print(f"   Perturbations per explanation: 5000")


In [ ]:
# Explain a single test sample
sample = X_test[0]
result: LIMEExplanation = lime_explainer.explain(model.predict_proba, sample, label=1)

print(f"Predicted class:     {result.predicted_class}")
print(f"Confidence:          {result.predicted_proba:.4f}")
print(f"Computation time:    {result.computation_time_ms:.1f} ms")
print(f"Local R²:            {result.local_r2:.4f}  (>0.5 = trustworthy surrogate)")
print(f"Intercept:           {result.intercept:.4f}")
print(f"\nTop 10 LIME Features:")
for i, fc in enumerate(result.top_k(10), 1):
    bar = "█" * int(abs(fc.contribution) * 50)
    sign = "+" if fc.contribution > 0 else "-"
    print(f"  {i:2}. {fc.feature_name:<15} {sign}{abs(fc.contribution):.4f}  {bar}")


---
## Subphase 40.2 — SHAP vs LIME Feature Ranking Correlation (Table 8)

We run SHAP and LIME on **200 identical samples** and compute Spearman ρ.
This produces **Table 8** of your research paper.

**Interpretation:**
- ρ > 0.8 → Strong agreement → Both methods trustworthy
- ρ 0.5–0.8 → Moderate agreement → Investigate where they disagree
- ρ < 0.5 → Weak agreement → Model relies on complex interactions LIME cannot capture


In [ ]:
import shap

N_COMPARE = min(50, len(X_test))  # 200 in paper; 50 here for speed
X_compare = X_test[:N_COMPARE]

print(f"Computing SHAP values for {N_COMPARE} samples...")
shap_explainer = shap.TreeExplainer(model)
shap_vals = shap_explainer.shap_values(X_compare)
if isinstance(shap_vals, list): shap_vals = np.array(shap_vals).mean(axis=0)
shap_importance = np.abs(shap_vals).mean(axis=0)  # mean |SHAP| per feature
print(f"✅ SHAP done. Shape: {shap_vals.shape}")

print(f"Computing LIME values for {N_COMPARE} samples...")
lime_importance = np.zeros(N_FEATURES)
for i, row in enumerate(X_compare):
    exp = lime_explainer.explain(model.predict_proba, row, label=1)
    for fc in exp.feature_contributions:
        idx = feature_names.index(fc.feature_name) if fc.feature_name in feature_names else -1
        if idx >= 0:
            lime_importance[idx] += fc.abs_contribution
    if (i+1) % 10 == 0: print(f"   LIME: {i+1}/{N_COMPARE}")
lime_importance /= N_COMPARE
print(f"✅ LIME done.")

# Spearman rank correlation between SHAP and LIME importance vectors
rho, pval = spearmanr(shap_importance, lime_importance)
print(f"\n=== Table 8 — SHAP vs LIME Spearman ρ ===")
print(f"  ρ = {rho:.4f}  (p = {pval:.4f})")
print(f"  Interpretation: {"Strong" if rho > 0.8 else "Moderate" if rho > 0.5 else "Weak"} agreement")


In [ ]:
os.makedirs("../../artifacts/figures", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: side-by-side bar chart
top_n = 15
top_idx = np.argsort(shap_importance)[::-1][:top_n]
x = np.arange(top_n)
s_norm = shap_importance[top_idx] / (shap_importance[top_idx].max() + 1e-9)
l_norm = lime_importance[top_idx] / (lime_importance[top_idx].max() + 1e-9)
axes[0].bar(x - 0.2, s_norm, 0.4, label="SHAP", color="#2196F3", alpha=0.85)
axes[0].bar(x + 0.2, l_norm, 0.4, label="LIME", color="#FF5722", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([feature_names[i][:8] for i in top_idx], rotation=45, ha="right", fontsize=8)
axes[0].set_title("SHAP vs LIME — Top 15 Features (normalised)", fontweight="bold")
axes[0].legend(); axes[0].set_ylabel("Normalised Importance")

# Right: scatter correlation
axes[1].scatter(s_norm, l_norm[:top_n], color="#9C27B0", s=60, alpha=0.8)
z = np.polyfit(s_norm, l_norm[:top_n], 1)
axes[1].plot(s_norm, np.poly1d(z)(s_norm), "r--", alpha=0.6, label=f"ρ={rho:.3f}")
axes[1].set_xlabel("SHAP importance (normalised)")
axes[1].set_ylabel("LIME importance (normalised)")
axes[1].set_title(f"Figure — SHAP vs LIME Correlation\nSpearman ρ = {rho:.3f}  p = {pval:.4f}", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../../artifacts/figures/shap_lime_correlation_p40.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ Figure saved to ml/artifacts/figures/shap_lime_correlation_p40.png")


In [ ]:
with mlflow.start_run(run_name="Phase40_LIME_vs_SHAP") as run:
    mlflow.log_metric("shap_lime_spearman_mean", float(rho))
    mlflow.log_metric("shap_lime_spearman_pval", float(pval))
    mlflow.log_metric("n_compare_samples", N_COMPARE)
    mlflow.log_param("n_perturbations", 5000)
    mlflow.log_param("random_state", 42)
    mlflow.log_artifact("../../artifacts/figures/shap_lime_correlation_p40.png")
print(f"✅ Logged to MLflow run: {run.info.run_id[:8]}")


---
## Subphase 40.3 — LIME Stability Testing

LIME is stochastic by design. We test how much explanations vary across 10 seeds.
**Threshold:** stability_score > 0.90 for production use.
If stability < 0.90, LIME cannot be trusted for individual analyst decisions.


In [ ]:
# LIMEStabilityTester needs a factory function (seed is baked into the explainer)
def lime_factory(seed: int) -> XAIGuardLIMEExplainer:
    return XAIGuardLIMEExplainer(
        X_train=X_train,
        feature_names=feature_names,
        n_perturbations=500,  # fewer for speed in testing; use 5000 in production
        random_state=seed,
    )

stability_tester = LIMEStabilityTester(lime_factory, n_runs=5)
X_stability = X_test[:20]

print("Running LIME stability test (5 seeds × 20 samples)...")
report: StabilityReport = stability_tester.run(X_stability, model.predict_proba, feature_names)

print("\n=== LIME Stability Report ===")
print(report.summary())

# Log to MLflow
with mlflow.start_run(run_name="Phase40_LIME_Stability") as run:
    mlflow.log_metric("lime_stability_score", report.stability_score)
    mlflow.log_metric("lime_stability_passed", int(report.passed))
print(f"✅ Stability logged to MLflow")


---
## ✅ Summary — Phase 40 — LIME Explainability

**What was built:**
- `ml/src/xai/lime_explainer.py` — `XAIGuardLIMEExplainer` with deterministic seed=42, n_perturbations=5000, `LIMEExplanation` dataclass with local R² faithfulness score
- `ml/src/xai/stability.py` — `LIMEStabilityTester` and `SHAPStabilityTester` using CoV metric
- This notebook: Phase 40.1 (single LIME explanation), 40.2 (SHAP-LIME Spearman ρ → Table 8), 40.3 (stability test)

**Key findings:**
- LIME computation time: ~2–5 seconds per explanation (within budget)
- SHAP-LIME Spearman ρ is logged to MLflow as `shap_lime_spearman_mean`
- LIME stability score is logged as `lime_stability_score`

**Next → Phase 41: Attention Rollout XAI for Transformer/LSTM**
